# Chapter 31 — MLOps: The Machine Learning Lifecycle in Production

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part5_production_rl/ch31_mlops/ch31_notebook.ipynb)

Companion notebook for **Chapter 31** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- MLOps combines machine learning with DevOps to automate the entire model lifecycle.
- A model that works in a notebook is not a product; production brings unpredictable data, traffic and scale.
- Track every experiment and version models, data and code together, or you cannot reproduce or debug anything.
- Monitor continuously for data drift and performance decay, and retrain on a schedule, on drift, or on degradation.
- Versioning plus a registry is what makes a fast, safe rollback possible when a new model misbehaves.

## Example 1: Experiment Tracking (What MLflow Does)

This builds a miniature MLflow: every run logs its parameters, metrics and an artifact hash, then all runs are compared side by side exactly as the MLflow UI shows them. Five model configurations are trained and the best is selected from the logs, and the train-test gap column shows how logged metrics let you spot overfitting without rerunning anything.

In [1]:
# Chapter 31 - Example 1: MLflow-style experiment tracking (from scratch)
import json, time, hashlib, os
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

np.random.seed(0)
print("=== Experiment Tracking (the core of MLflow) ===\n")
print("Without tracking you cannot answer: which run produced this model?")
print("what data and hyperparameters were used? can we reproduce it?\n")

STORE = "mlruns.json"   # written next to where you run the script
runs = []

def log_run(experiment, params, metrics, artifact_bytes):
    """Equivalent of mlflow.log_params / log_metrics / log_model."""
    run = {
        "run_id"    : hashlib.md5(f"{experiment}{params}".encode()).hexdigest()[:8],
        "experiment": experiment,
        "params"    : params,
        "metrics"   : metrics,
        "artifact_hash": hashlib.md5(artifact_bytes).hexdigest()[:12],
    }
    runs.append(run)
    return run

# ---- Data ----
data = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.3, random_state=42)
sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
print(f"Dataset: {data.data.shape[0]} samples, {data.data.shape[1]} features\n")

# ---- Run several experiments, logging each one ----
configs = [
    ("logreg",       LogisticRegression(C=0.1,  max_iter=2000, random_state=0), {"model":"LogisticRegression","C":0.1}),
    ("logreg",       LogisticRegression(C=1.0,  max_iter=2000, random_state=0), {"model":"LogisticRegression","C":1.0}),
    ("logreg",       LogisticRegression(C=10.0, max_iter=2000, random_state=0), {"model":"LogisticRegression","C":10.0}),
    ("random_forest",RandomForestClassifier(n_estimators=50,  random_state=0),  {"model":"RandomForest","n_estimators":50}),
    ("random_forest",RandomForestClassifier(n_estimators=200, random_state=0),  {"model":"RandomForest","n_estimators":200}),
]

print("Running experiments and logging every run...\n")
for exp, model, params in configs:
    model.fit(Xtr, ytr)
    pred_tr, pred_te = model.predict(Xtr), model.predict(Xte)
    metrics = {
        "train_accuracy": round(accuracy_score(ytr, pred_tr), 4),
        "test_accuracy" : round(accuracy_score(yte, pred_te), 4),
        "test_f1"       : round(f1_score(yte, pred_te), 4),
    }
    artifact = repr(getattr(model, "coef_", getattr(model, "feature_importances_", ""))).encode()
    r = log_run(exp, params, metrics, artifact)
    print(f"  run {r['run_id']} | {exp:<14} | {str(params):<48} | test_acc={metrics['test_accuracy']}")

with open(STORE, "w") as f:
    json.dump(runs, f, indent=2)

# ---- The "MLflow UI": compare runs ----
print(f"\n=== Run Comparison (what the MLflow UI shows) ===\n")
print(f"{'run_id':<10}{'experiment':<16}{'train_acc':>10}{'test_acc':>10}{'test_f1':>9}  {'artifact'}")
print("-" * 78)
for r in runs:
    print(f"{r['run_id']:<10}{r['experiment']:<16}"
          f"{r['metrics']['train_accuracy']:>10}{r['metrics']['test_accuracy']:>10}"
          f"{r['metrics']['test_f1']:>9}  {r['artifact_hash']}")

best = max(runs, key=lambda r: r["metrics"]["test_accuracy"])
print(f"\nBest run: {best['run_id']}  ({best['params']})")
print(f"  test accuracy = {best['metrics']['test_accuracy']}")

# ---- Overfitting check straight from the logs ----
print("\n=== Generalization check from logged metrics ===\n")
for r in runs:
    gap = r["metrics"]["train_accuracy"] - r["metrics"]["test_accuracy"]
    flag = "OVERFIT" if gap > 0.05 else "ok"
    print(f"  {r['run_id']}  train-test gap = {gap:+.4f}  [{flag}]")

print(f"\nAll runs persisted to {STORE} ({os.path.getsize(STORE)} bytes)")
print("This is exactly what MLflow does: log params, metrics and artifacts")
print("for every run so results are comparable, auditable and reproducible.")

=== Experiment Tracking (the core of MLflow) ===

Without tracking you cannot answer: which run produced this model?
what data and hyperparameters were used? can we reproduce it?

Dataset: 569 samples, 30 features

Running experiments and logging every run...

  run 5a0a2d82 | logreg         | {'model': 'LogisticRegression', 'C': 0.1}        | test_acc=0.9883
  run 612da96f | logreg         | {'model': 'LogisticRegression', 'C': 1.0}        | test_acc=0.9825
  run a35a4b4f | logreg         | {'model': 'LogisticRegression', 'C': 10.0}       | test_acc=0.9766
  run 526014c4 | random_forest  | {'model': 'RandomForest', 'n_estimators': 50}    | test_acc=0.9708
  run 821b0f28 | random_forest  | {'model': 'RandomForest', 'n_estimators': 200}   | test_acc=0.9649

=== Run Comparison (what the MLflow UI shows) ===

run_id    experiment       train_acc  test_acc  test_f1  artifact
------------------------------------------------------------------------------
5a0a2d82  logreg              0.9774 

## Example 2: Model Registry with Versioning and Rollback

This builds the registry concepts described above: register successive versions with their data hash and code commit, promote through Staging to Production, and - when v3 degrades in production - roll back to the last known-good version in one step. The audit trail at the end is exactly what a postmortem or a regulator would ask for.

In [2]:
# Chapter 31 - Example 2: Model registry with versioning, staging and rollback
import json, hashlib, datetime
import numpy as np

print("=== Model Registry: Versioning, Promotion and Rollback ===\n")
print("A registry answers: which model is in production RIGHT NOW, what data")
print("and code produced it, and how do we roll back if it misbehaves?\n")

class ModelRegistry:
    def __init__(self):
        self.versions = []          # all registered versions
        self.stages = {"Production": None, "Staging": None}
        self.audit = []

    def register(self, name, metrics, data_hash, code_commit):
        v = len(self.versions) + 1
        entry = {"name": name, "version": v, "metrics": metrics,
                 "data_hash": data_hash, "code_commit": code_commit,
                 "stage": "None",
                 "registered": f"2026-03-{10+v:02d}"}
        self.versions.append(entry)
        self.audit.append(f"registered v{v}")
        return entry

    def transition(self, version, stage):
        prev = self.stages.get(stage)
        if prev:
            self.versions[prev-1]["stage"] = "Archived"
        self.versions[version-1]["stage"] = stage
        self.stages[stage] = version
        self.audit.append(f"v{version} -> {stage}" + (f" (v{prev} archived)" if prev else ""))

    def show(self):
        print(f"  {'ver':<5}{'registered':<13}{'test_acc':>9}{'latency_ms':>12}  {'stage':<12}{'data':<10}{'commit'}")
        print("  " + "-" * 78)
        for e in self.versions:
            print(f"  v{e['version']:<4}{e['registered']:<13}{e['metrics']['test_acc']:>9}"
                  f"{e['metrics']['latency_ms']:>12}  {e['stage']:<12}{e['data_hash']:<10}{e['code_commit']}")

reg = ModelRegistry()

print("--- Registering successive model versions ---\n")
reg.register("churn_model", {"test_acc": 0.842, "latency_ms": 21}, "d1a3f9", "a91c4e2")
reg.register("churn_model", {"test_acc": 0.871, "latency_ms": 24}, "d1a3f9", "b32d81f")
reg.register("churn_model", {"test_acc": 0.889, "latency_ms": 26}, "e77b21", "c4419aa")
reg.show()

print("\n--- Promote v2 to Production ---")
reg.transition(2, "Production")
reg.show()

print("\n--- v3 looks better offline: promote to Staging, then Production ---")
reg.transition(3, "Staging")
reg.transition(3, "Production")
reg.show()

print("\n--- INCIDENT: v3 degrades in production (data mismatch) ---")
print("  Live monitoring: accuracy dropped to 0.71, error rate up 3x")
print("  Action: roll back to the last known-good version (v2)\n")
reg.transition(2, "Production")
reg.show()

print("\n--- Audit trail (what regulators / postmortems need) ---")
for i, a in enumerate(reg.audit, 1):
    print(f"  {i}. {a}")

print("\n=== Why versioning matters ===")
print("  * Compare: a model trained in January differs from one trained in March")
print("  * Roll back quickly to a stable model, minimizing downtime")
print("  * Reproducibility: model + data + code are versioned together")
print("  * Tools: Git (code), DVC / LakeFS (data), MLflow (models/metrics),")
print("           SageMaker Model Registry (cloud model versions)")

# Reproducibility check
print("\n=== Reproducibility check ===\n")
print("Re-running v2 with the SAME data hash and code commit should give")
print("the same metrics. If it does not, something untracked has changed.")
v2 = reg.versions[1]
rerun = {"test_acc": 0.871, "latency_ms": 24}
same = rerun == {k: v2["metrics"][k] for k in rerun}
print(f"  original v2 metrics : {v2['metrics']}")
print(f"  re-run metrics      : {rerun}")
print(f"  reproducible?       : {same}")

=== Model Registry: Versioning, Promotion and Rollback ===

A registry answers: which model is in production RIGHT NOW, what data
and code produced it, and how do we roll back if it misbehaves?

--- Registering successive model versions ---

  ver  registered    test_acc  latency_ms  stage       data      commit
  ------------------------------------------------------------------------------
  v1   2026-03-11       0.842          21  None        d1a3f9    a91c4e2
  v2   2026-03-12       0.871          24  None        d1a3f9    b32d81f
  v3   2026-03-13       0.889          26  None        e77b21    c4419aa

--- Promote v2 to Production ---
  ver  registered    test_acc  latency_ms  stage       data      commit
  ------------------------------------------------------------------------------
  v1   2026-03-11       0.842          21  None        d1a3f9    a91c4e2
  v2   2026-03-12       0.871          24  Production  d1a3f9    b32d81f
  v3   2026-03-13       0.889          26  None      

## Example 3: Drift Detection Triggering Retraining

This is the monitoring stage in code. It computes the Population Stability Index and a Kolmogorov-Smirnov test across four months of production data, tracks the model's accuracy decay against its deployment baseline, and then applies trigger logic combining drift, performance drop and elapsed time to decide automatically when retraining should fire.

In [3]:
# Chapter 31 - Example 3: Data/model drift detection triggering retraining
import numpy as np
from scipy import stats
np.random.seed(3)

print("=== Monitoring: Detecting Drift and Triggering Retraining ===\n")
print("A deployed model silently decays when the world changes. Monitoring")
print("must detect (a) INPUT DATA DRIFT and (b) MODEL PERFORMANCE DROP.\n")

# ---------- Population Stability Index ----------
def psi(expected, actual, bins=10):
    """PSI < 0.1 no shift | 0.1-0.25 moderate | > 0.25 major shift"""
    edges = np.percentile(expected, np.linspace(0, 100, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e_pct = np.histogram(expected, edges)[0] / len(expected)
    a_pct = np.histogram(actual,   edges)[0] / len(actual)
    e_pct = np.clip(e_pct, 1e-6, None); a_pct = np.clip(a_pct, 1e-6, None)
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

n = 2000
reference = np.random.normal(50, 10, n)      # training-time feature distribution

months = {
  "Month 1 (same as training)"  : np.random.normal(50.0, 10.0, n),
  "Month 2 (slight shift)"      : np.random.normal(51.5, 10.0, n),
  "Month 3 (moderate shift)"    : np.random.normal(54.0, 11.0, n),
  "Month 4 (major shift)"       : np.random.normal(60.0, 14.0, n),
}

print("--- Input Data Drift: PSI and Kolmogorov-Smirnov test ---\n")
print(f"{'Production window':<30}{'PSI':>8}{'KS stat':>10}{'p-value':>11}  {'Verdict'}")
print("-" * 76)
drift_flags = {}
for name, actual in months.items():
    p = psi(reference, actual)
    ks, pv = stats.ks_2samp(reference, actual)
    if p < 0.10:   verdict = "no significant drift"
    elif p < 0.25: verdict = "MODERATE drift - watch"
    else:          verdict = "MAJOR drift - retrain"
    drift_flags[name] = p
    print(f"  {name:<28}{p:>8.4f}{ks:>10.4f}{pv:>11.2e}  {verdict}")

# ---------- Model performance drift ----------
print("\n--- Model Performance Drift ---\n")
baseline_acc = 0.912
observed = {"Month 1": 0.909, "Month 2": 0.901, "Month 3": 0.874, "Month 4": 0.812}
threshold = 0.05
print(f"Baseline test accuracy at deployment: {baseline_acc}")
print(f"Alert threshold: relative drop > {threshold*100:.0f}%\n")
print(f"{'Window':<12}{'Accuracy':>10}{'Drop':>10}  {'Action'}")
print("-" * 56)
for m, acc in observed.items():
    drop = (baseline_acc - acc) / baseline_acc
    action = "TRIGGER RETRAINING" if drop > threshold else "continue monitoring"
    print(f"  {m:<10}{acc:>10.3f}{drop*100:>9.1f}%  {action}")

# ---------- Retraining decision logic ----------
print("\n--- Automated Retraining Trigger Logic ---\n")
def should_retrain(psi_value, acc_drop, days_since_train):
    reasons = []
    if psi_value > 0.25:      reasons.append(f"data drift PSI={psi_value:.3f}")
    if acc_drop > 0.05:       reasons.append(f"accuracy drop {acc_drop*100:.1f}%")
    if days_since_train > 30: reasons.append(f"scheduled ({days_since_train} days)")
    return (len(reasons) > 0), reasons

cases = [
    ("Month 1", 0.02, 0.003, 7),
    ("Month 3", 0.18, 0.042, 21),
    ("Month 4", 0.41, 0.110, 35),
]
for label, p, drop, days in cases:
    go, why = should_retrain(p, drop, days)
    status = "RETRAIN" if go else "no action"
    print(f"  {label}: PSI={p:.2f}, drop={drop*100:.1f}%, {days}d since training -> {status}")
    for r in why:
        print(f"      reason: {r}")

print("\n=== Retraining triggers ===")
print("  1. Scheduled intervals   - e.g. retail forecasting retrains weekly")
print("  2. Data / model drift    - ride-sharing model after a big city event")
print("  3. Performance drop      - fraud detection when errors spike")
print("\nTools: Prometheus (infrastructure), Evidently AI (data drift & ML")
print("metrics), CloudWatch (cloud apps); Airflow / Kubeflow run the pipeline.")

=== Monitoring: Detecting Drift and Triggering Retraining ===

A deployed model silently decays when the world changes. Monitoring
must detect (a) INPUT DATA DRIFT and (b) MODEL PERFORMANCE DROP.

--- Input Data Drift: PSI and Kolmogorov-Smirnov test ---

Production window                  PSI   KS stat    p-value  Verdict
----------------------------------------------------------------------------
  Month 1 (same as training)    0.0039    0.0255   5.34e-01  no significant drift
  Month 2 (slight shift)        0.0249    0.0595   1.68e-03  no significant drift
  Month 3 (moderate shift)      0.1646    0.1715   4.30e-26  MODERATE drift - watch
  Month 4 (major shift)         0.6911    0.3435  5.30e-105  MAJOR drift - retrain

--- Model Performance Drift ---

Baseline test accuracy at deployment: 0.912
Alert threshold: relative drop > 5%

Window        Accuracy      Drop  Action
--------------------------------------------------------
  Month 1        0.909      0.3%  continue monitoring
